# s07 — `web_search` + `web_fetch`

**What this teaches:** how to push the agent off the local machine and onto the open web. `web_search` returns a result list (DuckDuckGo by default, SerpAPI / Google when configured); `web_fetch` downloads a URL and returns it as markdown. Combine the two and the agent gains a primitive research loop.

**Concept anchor:** the model's training data has a knowledge cutoff. Pair it with `web_search`+`web_fetch` and it can answer questions about events that post-date its training — *if* you allow it network access. This is the cheapest possible RAG.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars (see [docs/providers.md](../../docs/providers.md)).
- **Outbound network access** — both tools call live HTTP endpoints. The notebook will hang or error if the network is locked down.
- **Optional:** `SERPAPI_KEY` set in the environment to back `web_search` with SerpAPI's Google engine. Without it the demo falls back to DuckDuckGo's free HTML endpoint.

## 1. Imports

Same trio as before — `fstools` exposes `NewWebTools`, `NewDDGTools`, and `NewSerpAPITools`.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/stream"
    fstools "github.com/blouargant/yoke/core/tools"
)

## 2. Helper

Panic-based `must` so the GoNB kernel survives transient HTTP errors.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Pick a search backend

`fstools.NewWebTools()` gives us `web_fetch` (and other HTTP helpers). We then append *either* `NewSerpAPITools(key)` *or* `NewDDGTools()` depending on whether a `SERPAPI_KEY` is in the environment. Both register a tool named `web_search` — only one is wired in per run.

In [ ]:
tools := fstools.NewWebTools()
if key := os.Getenv("SERPAPI_KEY"); key != "" {
    fmt.Fprintln(os.Stderr, "(web_search backed by SerpAPI / Google)")
    tools = append(tools, fstools.NewSerpAPITools(key)...)
} else {
    fmt.Fprintln(os.Stderr, "(web_search backed by DuckDuckGo — set SERPAPI_KEY for Google)")
    tools = append(tools, fstools.NewDDGTools()...)
}

## 4. Construct the model and agent

The instruction pins the research recipe: search → pick a URL → fetch → summarise with a citation. Without the instruction, models often hallucinate an answer without ever calling `web_fetch`.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)

a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s07_web_search",
    Description: "web_search + web_fetch demo.",
    Model:       llm,
    Instruction: "Use web_search to find relevant pages, then use web_fetch on " +
        "the most promising URL. Summarise the answer in 3 sentences citing the URL.",
    Tools: tools,
})
must(err)

## 5. Build the runner and run

Expect a `web_search` call, one or two `web_fetch` calls, then a short cited summary. Total latency is dominated by the HTTP round-trips, not by the model.

In [ ]:
r, err := agentkit.Runner("s07", a)
must(err)

prompt := "Who maintains the Go ADK library and what is its primary purpose?"
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- A `web_search` tool call followed by at least one `web_fetch` — if the model skips `web_fetch`, the answer is almost certainly hallucinated.
- The stderr banner from cell 3 confirms whether DuckDuckGo or SerpAPI is in use; rate limits differ wildly between them.
- For purely local tools (no network), see [s05_tools](../s05_tools/s05_tools.ipynb).

## Try it yourself

1. Ask about an event from the last month and verify the cited URL is reachable.
2. Set `SERPAPI_KEY` (if you have one) and re-run — compare result quality with the DuckDuckGo fallback.